In [8]:
# ---- Cell 1: imports + repo config ----
import pandas as pd
import json
import requests
from urllib.parse import quote

GITHUB_USER = "ChaitanyaDEV1243"
GITHUB_REPO = "flood-warning"
BRANCH = "main"
RAW_BASE = f"https://raw.githubusercontent.com/{GITHUB_USER}/{GITHUB_REPO}/{BRANCH}"

def raw_url(path):
    # percent-encode each path segment (handles spaces/parens) but keep the slashes
    return "/".join(quote(part) for part in path.split("/"))

def get_json(path):
    url = f"{RAW_BASE}/{raw_url(path)}"
    r = requests.get(url)
    if r.status_code != 200:
        raise FileNotFoundError(f"{url} -> {r.status_code}")
    return r.json()

def get_csv(path):
    url = f"{RAW_BASE}/{raw_url(path)}"
    return pd.read_csv(url)

In [9]:
# ---- Cell 2: Stage 1 — real paths, confirmed to exist ----
stage1_risk = get_json("stage1_gnn/data/stage1_output.json")
zone_df = get_csv("stage1_gnn/data/zone_features_final.csv")
model_comparison = get_csv("stage1_gnn/data/model_comparison.csv")

print(f"Loaded risk scores for {len(stage1_risk)} zones")
print("\nZone dataframe columns:", zone_df.columns.tolist())
print("\nModel comparison (baseline vs GCN vs GAT):")
model_comparison

Loaded risk scores for 11 zones

Zone dataframe columns: ['Rainfall (mm)', 'Temperature (°C)', 'Humidity (%)', 'River Discharge (m³/s)', 'Water Level (m)', 'Elevation (m)', 'Population Density', 'Historical Floods', 'Flood Occurred', 'zone_id', 'zone_name', 'lat', 'lon', 'district', 'STATE_UT_NAME', 'DISTRICT', 'JAN', 'FEB', 'MAR', 'APR', 'MAY', 'JUN', 'JUL', 'AUG', 'SEP', 'OCT', 'NOV', 'DEC', 'ANNUAL', 'Jan-Feb', 'Mar-May', 'Jun-Sep', 'Oct-Dec', 'fatalities', 'no_of_camps', 'actual_rainfall_in_mm', 'normal_rainfall_in_mm', 'no_of_landslides', 'full_damaged_houses']

Model comparison (baseline vs GCN vs GAT):


,model,MAE,R2
0,Baseline MLP (no spatial info),0.085316,-0.135051
1,GCN (spatial),0.058163,0.571302
2,GAT (spatial + attention),0.052992,0.650881


In [10]:
# ---- Cell 3: merge risk scores into zone_df ----
# Check the actual name column before mapping — print it first
print(zone_df.iloc[0])  # inspect one row to find the right column name

# Adjust "zone_name" below if the printed row shows a different column name
zone_df["risk_score"] = zone_df["zone_name"].str.strip().map(
    {k.strip(): v for k, v in stage1_risk.items()}
)
missing = zone_df[zone_df["risk_score"].isna()]
if len(missing):
    print("⚠️ Unmatched zones — check spelling/casing:", missing["zone_name"].tolist())

zone_df.sort_values("risk_score", ascending=False)

Rainfall (mm)               179.020074
Temperature (°C)             32.803759
Humidity (%)                  60.31237
River Discharge (m³/s)     2634.912132
Water Level (m)               5.042234
Elevation (m)              4471.362511
Population Density         4612.285232
Historical Floods                  0.4
Flood Occurred                     0.4
zone_id                            1.0
zone_name                 Periyar Lake
lat                           9.530984
lon                           77.18964
district                        Idukki
STATE_UT_NAME                      NaN
DISTRICT                           NaN
JAN                                NaN
FEB                                NaN
MAR                                NaN
APR                                NaN
MAY                                NaN
JUN                                NaN
JUL                                NaN
AUG                                NaN
SEP                                NaN
OCT                      

,Rainfall (mm),Temperature (°C),Humidity (%),River Discharge (m³/s),Water Level (m),Elevation (m),Population Density,Historical Floods,Flood Occurred,zone_id,...,Mar-May,Jun-Sep,Oct-Dec,fatalities,no_of_camps,actual_rainfall_in_mm,normal_rainfall_in_mm,no_of_landslides,full_damaged_houses,risk_score
9,157.662451,32.781690,56.548461,2663.278605,4.846719,4032.214937,4513.328567,0.35,0.55,10.0,...,NaN,NaN,NaN,58,1582,648.3,401.3,0,615,0.4929
10,151.660939,30.264647,56.395526,2926.067000,5.244574,3946.131203,4822.628050,0.40,0.50,11.0,...,NaN,NaN,NaN,58,1582,648.3,401.3,0,615,0.4806
8,157.662451,32.781690,56.548461,2663.278605,4.846719,4032.214937,4513.328567,0.35,0.55,9.0,...,NaN,NaN,NaN,58,1582,648.3,401.3,0,615,0.4774
7,149.286991,33.758767,56.388206,2722.207403,5.099284,4065.939584,4691.249715,0.30,0.50,8.0,...,NaN,NaN,NaN,58,1582,648.3,401.3,0,615,0.4617
6,152.610791,32.946206,56.951601,2671.812266,5.232067,3842.456288,4986.218594,0.25,0.50,7.0,...,NaN,NaN,NaN,58,1582,648.3,401.3,0,615,0.4381
5,139.856770,33.682020,53.987643,2613.199721,5.727455,4320.834076,5648.181619,0.30,0.40,6.0,...,NaN,NaN,NaN,58,1582,648.3,401.3,0,615,0.4095
0,179.020074,32.803759,60.312370,2634.912132,5.042234,4471.362511,4612.285232,0.40,0.40,1.0,...,NaN,NaN,NaN,54,363,1478.9,527.3,143,1166,0.3756
1,176.295573,33.106101,60.653973,2655.926103,5.159388,4647.635172,5015.310203,0.40,0.35,2.0,...,NaN,NaN,NaN,54,363,1478.9,527.3,143,1166,0.3705
4,148.952103,33.336989,63.307913,2495.150028,5.417636,4285.178421,6040.134009,0.30,0.30,5.0,...,NaN,NaN,NaN,54,363,1478.9,527.3,143,1166,0.3424
2,158.503117,33.422386,58.236913,2718.855050,4.755312,4048.099005,5546.121252,0.35,0.30,3.0,...,NaN,NaN,NaN,54,363,1478.9,527.3,143,1166,0.3376


In [11]:
# ---- Cell 4: Stage 2 — CORRECTED path (extra nested "flood-warning/" folder) ----
rl_allocation = get_json("flood-warning/stage2_allocation/rl_allocation_result.json")
print(json.dumps(rl_allocation, indent=2))

{
  "model": "RL Agent (PPO)",
  "allocation_order": [
    "Vembanad Lake",
    "Aluva",
    "Neriamangalam",
    "Idukki Reservoir",
    "Munambam"
  ],
  "zones_covered": 5
}


In [12]:
# ---- Cell 5: Stage 2 baseline/comparison — this is a CSV, not JSON ----
comparison_df = get_csv("comparison_results (2).csv")
print(comparison_df.columns.tolist())
comparison_df

['Method', 'Coverage']


,Method,Coverage
0,Linear Programming,11205.717423
1,Greedy,11205.717423


In [13]:
# ---- Cell 6: diagnose the NaN rainfall columns ----
rainfall_cols = ["STATE_UT_NAME", "DISTRICT", "JAN", "FEB", "MAR", "APR", "MAY", "JUN",
                  "JUL", "AUG", "SEP", "OCT", "NOV", "DEC", "ANNUAL",
                  "Jan-Feb", "Mar-May", "Jun-Sep", "Oct-Dec"]

print("NaN count per column (out of", len(zone_df), "rows):")
print(zone_df[rainfall_cols].isna().sum())

print("\nUnique districts in zone_df:", zone_df["district"].unique())

NaN count per column (out of 11 rows):
STATE_UT_NAME    11
DISTRICT         11
JAN              11
FEB              11
MAR              11
APR              11
MAY              11
JUN              11
JUL              11
AUG              11
SEP              11
OCT              11
NOV              11
DEC              11
ANNUAL           11
Jan-Feb          11
Mar-May          11
Jun-Sep          11
Oct-Dec          11
dtype: int64

Unique districts in zone_df: ['Idukki' 'Ernakulam']


In [14]:
# ---- Cell 7: try to recover the greedy/LP allocation lists from the notebook ----
nb = get_json("stage2_lp_greedy_baseline_ipyn.ipynb")  # .ipynb is valid JSON

found_outputs = []
for cell in nb.get("cells", []):
    if cell.get("cell_type") != "code":
        continue
    src = "".join(cell.get("source", []))
    for out in cell.get("outputs", []):
        text = "".join(out.get("text", [])) if "text" in out else ""
        data_text = ""
        if "data" in out and "text/plain" in out["data"]:
            data_text = "".join(out["data"]["text/plain"])
        combined = text + data_text
        if any(z in combined for z in ["Periyar", "Idukki", "Aluva", "Munambam", "Vembanad", "Neriamangalam"]):
            found_outputs.append({"source": src[:200], "output": combined[:500]})

print(f"Found {len(found_outputs)} candidate outputs mentioning zone names:\n")
for f in found_outputs:
    print("SOURCE:", f["source"])
    print("OUTPUT:", f["output"])
    print("-" * 60)

Found 9 candidate outputs mentioning zone names:

SOURCE: print(df["zone_name"].tolist())
OUTPUT: ['Periyar Lake', 'Periyar-Vandiperiyar', 'Elappara', 'Ayyappancoil', 'Idukki Reservoir', 'Neriamangalam', 'Aluva', 'Udhyogamandal', 'Varapuzha', 'Vembanad Lake', 'Munambam']

------------------------------------------------------------
SOURCE: print("Zones:", len(zones))
print(zones)

print("\nPopulation:", len(population))
print(population)
OUTPUT: Zones: 11
['Periyar Lake', 'Periyar-Vandiperiyar', 'Elappara', 'Ayyappancoil', 'Idukki Reservoir', 'Neriamangalam', 'Aluva', 'Udhyogamandal', 'Varapuzha', 'Vembanad Lake', 'Munambam']

Population: 11
[4612.28523151 5015.31020317 5546.12125235 5505.60145305 6040.13400882
 5648.1816188  4986.21859356 4691.24971496 4513.32856657 4513.32856657
 4822.62804954]

------------------------------------------------------------
SOURCE: print(df["zone_name"].tolist())
OUTPUT: ['Periyar Lake', 'Periyar-Vandiperiyar', 'Elappara', 'Ayyappancoil', 'Idukki Reser

In [15]:
# ---- Cell 8: consolidate everything into clean, dashboard-ready files ----
import json

# Baseline allocation (recovered from notebook outputs — not available as JSON in repo)
baseline_allocation = {
    "method": "Greedy / Linear Programming (converged on same result)",
    "allocation_order": ["Neriamangalam", "Aluva", "Udhyogamandal", "Vembanad Lake", "Munambam"],
    "zones_covered": 5,
    "coverage_value": 11205.717423  # from comparison_results (2).csv
}

rl_allocation_clean = {
    "method": "RL Agent (PPO)",
    "allocation_order": rl_allocation["allocation_order"],
    "zones_covered": rl_allocation["zones_covered"]
}

# The zone that differs, for your dashboard's "what changed" callout
baseline_set = set(baseline_allocation["allocation_order"])
rl_set = set(rl_allocation_clean["allocation_order"])
swapped_out = baseline_set - rl_set
swapped_in = rl_set - baseline_set

print("Baseline-only zone(s):", swapped_out)
print("RL-only zone(s):", swapped_in)

allocation_comparison = {
    "baseline": baseline_allocation,
    "rl": rl_allocation_clean,
    "diff": {
        "swapped_out": list(swapped_out),
        "swapped_in": list(swapped_in)
    }
}

with open("allocation_comparison.json", "w") as f:
    json.dump(allocation_comparison, f, indent=2)

# Drop the dead NaN rainfall columns from zone_df — we'll load real rainfall data separately
dead_cols = ["STATE_UT_NAME", "DISTRICT", "JAN", "FEB", "MAR", "APR", "MAY", "JUN",
             "JUL", "AUG", "SEP", "OCT", "NOV", "DEC", "ANNUAL",
             "Jan-Feb", "Mar-May", "Jun-Sep", "Oct-Dec"]
zone_df_clean = zone_df.drop(columns=dead_cols)
zone_df_clean.to_csv("zones_with_risk_clean.csv", index=False)

print("\nSaved: allocation_comparison.json, zones_with_risk_clean.csv")
zone_df_clean.head()

Baseline-only zone(s): {'Udhyogamandal'}
RL-only zone(s): {'Idukki Reservoir'}

Saved: allocation_comparison.json, zones_with_risk_clean.csv


,Rainfall (mm),Temperature (°C),Humidity (%),River Discharge (m³/s),Water Level (m),Elevation (m),Population Density,Historical Floods,Flood Occurred,zone_id,...,lat,lon,district,fatalities,no_of_camps,actual_rainfall_in_mm,normal_rainfall_in_mm,no_of_landslides,full_damaged_houses,risk_score
0,179.020074,32.803759,60.312370,2634.912132,5.042234,4471.362511,4612.285232,0.40,0.40,1.0,...,9.530984,77.189640,Idukki,54,363,1478.9,527.3,143,1166,0.3756
1,176.295573,33.106101,60.653973,2655.926103,5.159388,4647.635172,5015.310203,0.40,0.35,2.0,...,9.572483,77.089653,Idukki,54,363,1478.9,527.3,143,1166,0.3705
2,158.503117,33.422386,58.236913,2718.855050,4.755312,4048.099005,5546.121252,0.35,0.30,3.0,...,9.635681,76.978890,Idukki,54,363,1478.9,527.3,143,1166,0.3376
3,163.583666,32.813564,57.573909,2466.045114,4.819116,4327.362972,5505.601453,0.45,0.25,4.0,...,9.693551,77.047297,Idukki,54,363,1478.9,527.3,143,1166,0.3008
4,148.952103,33.336989,63.307913,2495.150028,5.417636,4285.178421,6040.134009,0.30,0.30,5.0,...,9.843056,76.976389,Idukki,54,363,1478.9,527.3,143,1166,0.3424


In [18]:
from google.colab import drive
drive.mount('/content/drive')
district_df = pd.read_csv('/content/drive/MyDrive/flood_project_data/district_wise_details.csv')
rainfall_df = pd.read_csv('/content/drive/MyDrive/flood_project_data/warnings_actual_predicted.csv')

Mounted at /content/drive


In [20]:
# ---- Cell 10a: diagnose — what actually happened? ----
import os

print("Files in current working directory (/content):")
print(os.listdir("."))

print("\nIs Drive mounted?", os.path.exists("/content/drive"))
if os.path.exists("/content/drive"):
    print("\nDrive contents (top level of MyDrive):")
    print(os.listdir("/content/drive/MyDrive"))

Files in current working directory (/content):
['.config', 'drive', 'zones_with_risk_clean.csv', 'allocation_comparison.json', 'sample_data']

Is Drive mounted? True

Drive contents (top level of MyDrive):
['NPS (6)', 'NPS (5)', 'NPS (4)', 'NPS (3)', 'rectangle.drawio', 'Screenshot_2024-09-14-11-52-10-72_40deb401b9ffe8e1df2f1cc5ba480b12.jpg', 'VID-20241114-WA0015.mp4', 'Nukkad Natak Assignment - 1', 'Priyanshu Kaushik 590015022 Assignment-2 Problem solving.pdf', 'Copy of Resume CV of Students - Srijan 4.0.gdoc', 'Priyanshu Kaushik 590015022 CV (4)', 'Priyanshu Kaushik 590015022 CV (3)', 'Priyanshu Kaushik 590015022 CV (2)', 'Document from Priyanshu Kaushik (1)', 'Priyanshu Kaushik 590015022 CV (1)', 'Priyanshu Kaushik 590015022 CV', 'Priyanshu Kaushik 590015022 CV.pdf', '590015022_BTECH_CS_CSE_SEM_1_SEM_GRADE_CARD_1_1740726389013_STD.pdf', 'NPS (2)', 'Copy of NPS', 'NPS (1)', 'Screenshot_2025-03-21-15-04-23-81_40deb401b9ffe8e1df2f1cc5ba480b12.jpg', 'NPS2.pdf', 'Priyanshu Kaushik 590015

In [21]:
# ---- Cell 10b: look inside flood_project_data ----
import os

target_folder = "/content/drive/MyDrive/flood_project_data"
print("Contents of flood_project_data:")
for item in os.listdir(target_folder):
    print(" ", item)

Contents of flood_project_data:
  rainfall in india 1901-2015.csv
  district wise rainfall normal.csv
  warnings_actual_predicted.csv
  warning.jpg
  district_wise_details.csv
  flood.csv
  flood_risk_dataset_india.csv
  archive (4).zip
  model_comparison.csv
  model_comparison_chart.png
  final_gat_model.pt
  stage1_output.json
  zone_features_final.csv
  Untitled0.ipynb


In [24]:
# ---- Cell 10c: load the two validation CSVs ----
import pandas as pd

target_folder = "/content/drive/MyDrive/flood_project_data"

district_df = pd.read_csv(f"{target_folder}/district_wise_details.csv")
rainfall_df = pd.read_csv(f"{target_folder}/warnings_actual_predicted.csv")

print("=== district_wise_details.csv ===")
print("Columns:", district_df.columns.tolist())
print("Shape:", district_df.shape)
print(district_df.head())

print("\n=== warnings_actual_predicted.csv ===")
print("Columns:", rainfall_df.columns.tolist())
print("Shape:", rainfall_df.shape)
print(rainfall_df.head())

=== district_wise_details.csv ===
Columns: ['district', 'fatalities', 'no_of_camps', 'actual_rainfall_in_mm', 'normal_rainfall_in_mm', 'no_of_landslides', 'full_damaged_houses']
Shape: (14, 7)
             district  fatalities  no_of_camps  actual_rainfall_in_mm  \
0  Thiruvananthapuram          11           94                  373.8   
1              Kollam           5          168                  644.1   
2      Pathanamthitta           3         4352                  764.9   
3           Alappuzha          43         2126                  608.2   
4            Kottayam          14          788                  619.2   

   normal_rainfall_in_mm  no_of_landslides  full_damaged_houses  
0                  142.0                 0                  111  
1                  258.7                 2                   95  
2                  352.7                 8                  741  
3                  343.1                 0                 2075  
4                  386.0              

In [25]:
# ---- Cell 10d: check district name formatting BEFORE we try to join anything ----
# Find whichever column holds district names in each file — print all columns with "dist" in the name
dist_cols_district_df = [c for c in district_df.columns if "dist" in c.lower()]
dist_cols_rainfall_df = [c for c in rainfall_df.columns if "dist" in c.lower()]

print("district_df district-like columns:", dist_cols_district_df)
for c in dist_cols_district_df:
    print(f"  Unique values in '{c}':", district_df[c].unique()[:20])

print("\nrainfall_df district-like columns:", dist_cols_rainfall_df)
for c in dist_cols_rainfall_df:
    print(f"  Unique values in '{c}':", rainfall_df[c].unique()[:20])

print("\nOur zone_df districts (for reference):", zone_df_clean["district"].unique())

district_df district-like columns: ['district']
  Unique values in 'district': ['Thiruvananthapuram' 'Kollam' 'Pathanamthitta' 'Alappuzha' 'Kottayam'
 'Idukki' 'Ernakulam' 'Thrissur' 'Palakkad' 'Malappuram' 'Kozhikode'
 'Wayanad' 'Kannur' 'Kasaragode']

rainfall_df district-like columns: ['district']
  Unique values in 'district': ['Alappuzha' 'Ernakulam' 'Idukki' 'Kannur' 'Kasaragod' 'Kollam' 'Kottayam'
 'Kozhikkode' 'Malappuram' 'Palakkad' 'Pathanamthitta'
 'Thiruvananthapuram' 'Thrissur' 'Wayanad']

Our zone_df districts (for reference): ['Idukki' 'Ernakulam']


In [26]:
# ---- Cell 11: district-level outcome validation (fatalities/damage vs our risk scores) ----
# zone_df_clean already has fatalities/camps/landslides/damage merged in from Person A's pipeline
# (confirmed working, unlike the dead JAN-DEC columns we dropped)

district_summary = (
    zone_df_clean.groupby("district")
    .agg(
        avg_risk_score=("risk_score", "mean"),
        max_risk_score=("risk_score", "max"),
        n_zones=("zone_name", "count"),
        fatalities=("fatalities", "first"),      # same value repeated per zone, so "first" is safe
        no_of_camps=("no_of_camps", "first"),
        no_of_landslides=("no_of_landslides", "first"),
        full_damaged_houses=("full_damaged_houses", "first"),
    )
    .reset_index()
)
print(district_summary)

    district  avg_risk_score  max_risk_score  n_zones  fatalities  \
0  Ernakulam        0.460033          0.4929        6          58   
1     Idukki        0.345380          0.3756        5          54   

   no_of_camps  no_of_landslides  full_damaged_houses  
0         1582                 0                  615  
1          363               143                 1166  


In [27]:
# ---- Cell 12: rainfall warning classification accuracy, Idukki + Ernakulam only ----
warning_order = {"Green": 0, "Yellow": 1, "Orange": 2, "Red": 3}

kerala_2018 = rainfall_df[rainfall_df["district"].isin(["Idukki", "Ernakulam"])].copy()
kerala_2018["actual_level"] = kerala_2018["actual_rainfall"].map(warning_order)
kerala_2018["predicted_level"] = kerala_2018["predicted_rainfall"].map(warning_order)
kerala_2018["exact_match"] = kerala_2018["actual_level"] == kerala_2018["predicted_level"]
kerala_2018["off_by"] = (kerala_2018["predicted_level"] - kerala_2018["actual_level"]).abs()

print(f"Rows for Idukki + Ernakulam: {len(kerala_2018)}")
print(f"Exact match rate: {kerala_2018['exact_match'].mean():.1%}")
print(f"Mean absolute warning-level error: {kerala_2018['off_by'].mean():.2f} levels (0=exact, 3=max)")
print(f"\nPer-district breakdown:")
print(kerala_2018.groupby("district")[["exact_match", "off_by"]].mean())

Rows for Idukki + Ernakulam: 30
Exact match rate: 43.3%
Mean absolute warning-level error: 0.93 levels (0=exact, 3=max)

Per-district breakdown:
           exact_match    off_by
district                        
Ernakulam     0.333333  1.133333
Idukki        0.533333  0.733333


In [29]:
# ---- Cell 13: save validation outputs + honest limitations note ----
import json

validation_summary = {
    "district_outcomes": district_summary.to_dict(orient="records"),
    "rainfall_warning_accuracy": {
        "exact_match_rate": 0.433,
        "mean_absolute_level_error": 0.93,
        "n_observations": 30,
        "per_district": {
            "Ernakulam": {"exact_match": 0.333, "mean_off_by": 1.133},
            "Idukki": {"exact_match": 0.533, "mean_off_by": 0.733}
        }
    },
    "limitations": [
        "Our GNN predicts risk at 11 zone-level points; ground truth (fatalities, damaged houses, landslides) exists only at district granularity, so validation is zone-aggregated-to-district, not zone-to-zone.",
        "Only 2 districts (Idukki, Ernakulam) are covered by our 11 zones, so any correlation between avg zone risk and district outcomes is illustrative (n=2), not statistically significant.",
        "Rainfall validation uses categorical warning levels (Green/Yellow/Orange/Red), not continuous rainfall values, despite the source column names.",
        "This is a retrospective comparison against the 2018 event, not a live/prospective validation — the model was not making real-time predictions during the actual event.",
        "43.3% exact-match should be read as 'directionally useful classification tendency,' not forecasting precision — mean error of ~1 warning tier is the more honest headline number."
    ]
}

district_summary.to_csv("district_validation_summary.csv", index=False)
kerala_2018.to_csv("rainfall_warning_validation.csv", index=False)
with open("validation_summary.json", "w") as f:
    json.dump(validation_summary, f, indent=2)

print("Saved: district_validation_summary.csv, rainfall_warning_validation.csv, validation_summary.json")

Saved: district_validation_summary.csv, rainfall_warning_validation.csv, validation_summary.json
